### Import libraries

In [2]:
import pandas as pd
import numpy as np

### 2.Load the raw dataset

In [3]:
df = pd.read_csv("../data/raw/customer_contacts_dirty.csv")

df.head()

,customer_id,first_name,last_name,email,phone,company,city,country,signup_date,status
0,C0001,John,Smith,john.smith@example.com,-1070,Acme Inc,New York,USA,15-01-2024,Active
1,C0002,Sarah,Johnson,sarah.j@example.com,4155550102,BlueSky Ltd,Los Angeles,USA,01/22/2024,active
2,C0003,Michael,Brown,michael.brown@example.com,+1 415 555 0103,TechNova,Chicago,USA,03-02-2024,ACTIVE
3,C0004,Emily,Davis,emily.davis@example.com,NaN,GreenLeaf,Seattle,USA,12-02-2024,Inactive
4,C0005,David,Wilson,david.wilson@example.com,555-0105,CloudNine,Austin,USA,01-03-2024,Active


### 3.Understand the dataset

In [4]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Shape: (150, 10)

Column names:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'company', 'city', 'country', 'signup_date', 'status']

Data types:
customer_id    str
first_name     str
last_name      str
email          str
phone          str
company        str
city           str
country        str
signup_date    str
status         str
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    str  
 1   first_name   150 non-null    str  
 2   last_name    150 non-null    str  
 3   email        149 non-null    str  
 4   phone        147 non-null    str  
 5   company      150 non-null    str  
 6   city         150 non-null    str  
 7   country      150 non-null    str  
 8   signup_date  150 non-null    str  
 9   status       150 non-null    str  
dtypes: str(10)
memory usage: 11.8 KB


### 4.Inspect missing values

In [5]:
missing_values = df.isnull().sum()

print(missing_values)

customer_id    0
first_name     0
last_name      0
email          1
phone          3
company        0
city           0
country        0
signup_date    0
status         0
dtype: int64


In [6]:
missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

missing_report

,missing_count,missing_percentage
customer_id,0,0.00
first_name,0,0.00
last_name,0,0.00
email,1,0.67
phone,3,2.00
company,0,0.00
city,0,0.00
country,0,0.00
signup_date,0,0.00
status,0,0.00


### 5.Inspect duplicates

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


In [8]:
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

Duplicate customer IDs: 0


### 6.Inspect all unique values in important columns

In [9]:
columns_to_inspect = ["city", "country", "status"]

for column in columns_to_inspect:
    print(f"\n--- {column.upper()} ---")
    print(df[column].unique())


--- CITY ---
<StringArray>
[   'New York', 'Los Angeles',     'Chicago',     'Seattle',      'Austin',
      'Boston',      'Denver',       'Miami',    'Portland',   'Bengaluru',
      'Mumbai',       'Delhi',   'Hyderabad',     'Chennai',        'Pune',
      'Jaipur',      'London',  'Manchester',     'Bristol']
Length: 19, dtype: str

--- COUNTRY ---
<StringArray>
['USA', 'India', 'UK', 'United Kingdom']
Length: 4, dtype: str

--- STATUS ---
<StringArray>
['Active', 'active', 'ACTIVE', 'Inactive', 'Pending', 'Unknown']
Length: 6, dtype: str


### 7.Detect leading/trailing whitespace

In [10]:
text_columns = df.select_dtypes(include="string").columns

whitespace_report = {}

for column in text_columns:
    mask = df[column].notna() & (
        df[column] != df[column].str.strip()
    )

    whitespace_report[column] = mask.sum()

whitespace_report

{'customer_id': np.int64(0),
 'first_name': np.int64(5),
 'last_name': np.int64(1),
 'email': np.int64(0),
 'phone': np.int64(0),
 'company': np.int64(0),
 'city': np.int64(0),
 'country': np.int64(0),
 'signup_date': np.int64(0),
 'status': np.int64(0)}

### 8.email validation

In [11]:
import re

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

email_check = df.copy()

email_check["email_valid"] = email_check["email"].apply(
    lambda email: bool(re.match(email_pattern, email.strip()))
    if pd.notna(email) and email.strip() != ""
    else False
)

invalid_emails = email_check[
    email_check["email_valid"] == False
]

invalid_emails[["customer_id", "first_name", "last_name", "email"]]

,customer_id,first_name,last_name,email
5,C0006,Jessica,Miller,jessica.miller@example
47,C0048,Michelle,Collins,not-an-email
75,C0076,,Martin,NaN


### 9.find duplicate emails 

In [12]:
df["email_normalized"] = (
    df["email"]
    .str.strip()
    .str.lower()
)

duplicate_email_records = df[
    df["email_normalized"].notna()
    & df["email_normalized"].duplicated(keep=False)
].sort_values("email_normalized")

duplicate_email_records[
    ["customer_id", "first_name", "last_name", "email", "email_normalized"]
]

,customer_id,first_name,last_name,email,email_normalized
103,C0104,Ananya,Reddy,ananya.reddy@example.com,ananya.reddy@example.com
128,C0129,Ananya,Reddy,ananya.reddy@example.com,ananya.reddy@example.com
100,C0101,Arjun,Sharma,arjun.sharma@example.com,arjun.sharma@example.com
120,C0121,Arjun,Sharma,arjun.sharma@example.com,arjun.sharma@example.com
67,C0068,Jessica,Wood,jessica.wood@example.com,jessica.wood@example.com
87,C0088,Jessica,Wood,jessica.wood@example.com,jessica.wood@example.com
41,C0042,Jonathan,Phillips,jonathan.phillips@example.com,jonathan.phillips@example.com
96,C0097,Jonathan,Phillips,jonathan.phillips@example.com,jonathan.phillips@example.com
42,C0043,Raymond,Campbell,raymond.campbell@example.com,raymond.campbell@example.com
99,C0100,Raymond,Campbell,raymond.campbell@example.com,raymond.campbell@example.com


### 10.Inspect phone formats

In [13]:
phone_inspection = df[["customer_id", "phone"]].copy()

phone_inspection["phone_digits"] = (
    phone_inspection["phone"]
    .str.replace(r"\D", "", regex=True)
)

phone_inspection["phone_length"] = (
    phone_inspection["phone_digits"].str.len()
)

phone_inspection.head(20)

,customer_id,phone,phone_digits,phone_length
0,C0001,-1070,1070,4.0
1,C0002,4155550102,4155550102,10.0
2,C0003,+1 415 555 0103,14155550103,11.0
3,C0004,NaN,NaN,NaN
4,C0005,555-0105,5550105,7.0
5,C0006,415-555-0106,4155550106,10.0
6,C0007,-1076,1076,4.0
7,C0008,415.555.0108,4155550108,10.0
8,C0009,4155550109,4155550109,10.0
9,C0010,+1 415 555 0110,14155550110,11.0


### 10.Investigate unusual phone lengths

In [14]:
phone_inspection["phone_length"].value_counts(dropna=False)

phone_length
10.0    131
12.0      7
6.0       4
NaN       3
4.0       2
11.0      2
7.0       1
Name: count, dtype: int64

### 12.Inspect the raw date values

In [15]:
df["signup_date"].value_counts().head(30)

signup_date
01-03-2024    3
01/22/2024    2
20-03-2024    2
01-04-2024    2
15-04-2024    2
01-05-2024    2
01-06-2024    2
01-07-2024    2
15-07-2024    2
01-08-2024    2
15-08-2024    2
01-09-2024    2
10-09-2024    2
01-10-2024    2
01-11-2024    2
15-11-2024    2
01-12-2024    2
10-12-2024    2
01-02-2025    2
15-02-2025    2
01-03-2025    2
10-03-2025    2
10-04-2025    2
10-05-2025    2
10-06-2025    2
10-07-2025    2
10-08-2025    2
10-11-2025    2
05-01-2024    2
15-01-2024    1
Name: count, dtype: int64

### 13.Attempt date parsing

In [16]:
parsed_dates = pd.to_datetime(
    df["signup_date"],
    errors="coerce",
    format="mixed"
)

date_check = pd.DataFrame({
    "original_date": df["signup_date"],
    "parsed_date": parsed_dates
})

date_check.head(30)

,original_date,parsed_date
0,15-01-2024,2024-01-15
1,01/22/2024,2024-01-22
2,03-02-2024,2024-03-02
3,12-02-2024,2024-12-02
4,01-03-2024,2024-01-03
5,15-03-2024,2024-03-15
6,20-03-2024,2024-03-20
7,01-04-2024,2024-01-04
8,08-04-2024,2024-08-04
9,15-04-2024,2024-04-15


### 14.Identify dates that failed parsing

In [17]:
invalid_dates = date_check[
    date_check["parsed_date"].isna()
]

invalid_dates

,original_date,parsed_date


### 15.Inspect customer status

In [18]:
print("Unique statuses:")
print(df["status"].value_counts(dropna=False))

Unique statuses:
status
Active      75
Inactive    37
Pending     27
active       8
ACTIVE       2
Unknown      1
Name: count, dtype: int64


### 16.Find possible duplicate customers

In [19]:
df["name_key"] = (
    df["first_name"].fillna("").str.strip().str.lower()
    + " "
    + df["last_name"].fillna("").str.strip().str.lower()
)

potential_duplicate_names = df[
    df["name_key"].duplicated(keep=False)
].sort_values("name_key")

potential_duplicate_names[
    ["customer_id", "first_name", "last_name", "email", "phone", "company"]
]

,customer_id,first_name,last_name,email,phone,company
128,C0129,Ananya,Reddy,ananya.reddy@example.com,9876543029,Bright Solutions
103,C0104,Ananya,Reddy,ananya.reddy@example.com,NaN,Bright Solutions
100,C0101,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs
120,C0121,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs
67,C0068,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf
87,C0088,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf
41,C0042,Jonathan,Phillips,jonathan.phillips@example.com,4155550142,NorthStar
96,C0097,Jonathan,Phillips,jonathan.phillips@example.com,4155550197,CloudNine
47,C0048,Michelle,Collins,not-an-email,4155550148,BlueSky Ltd
97,C0098,Michelle,Collins,michelle.collins@example.com,4155550198,NorthStar


## phase 2

### 17.Create a fresh working copy

In [20]:
df_raw = pd.read_csv("../data/raw/customer_contacts_dirty.csv")

df_clean = df_raw.copy()

print("Raw shape:", df_raw.shape)
print("Working shape:", df_clean.shape)

Raw shape: (150, 10)
Working shape: (150, 10)


### 18.Standardize text fields

In [21]:
# Remove leading and trailing whitespace
text_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "company",
    "city",
    "country",
    "status"
]

for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()

### 19.Standardize name capitalization

In [22]:
df_clean["first_name"] = df_clean["first_name"].str.title()
df_clean["last_name"] = df_clean["last_name"].str.title()

### 20.Standardize email addresses

In [23]:
df_clean["email"] = df_clean["email"].str.lower()

### 21.Standardize countries

In [24]:
country_mapping = {
    "USA": "United States",
    "UK": "United Kingdom",
    "United Kingdom": "United Kingdom",
    "India": "India"
}

df_clean["country"] = df_clean["country"].replace(country_mapping)

In [25]:
df_clean["country"].value_counts()

country
United States     100
India              30
United Kingdom     20
Name: count, dtype: int64

### 22.Standardize customer statuses 

In [26]:
status_mapping = {
    "active": "Active",
    "inactive": "Inactive",
    "pending": "Pending",
    "unknown": "Unknown"
}

df_clean["status"] = (
    df_clean["status"]
    .str.lower()
    .map(status_mapping)
)

In [27]:
df_clean["status"].value_counts(dropna=False)

status
Active      85
Inactive    37
Pending     27
Unknown      1
Name: count, dtype: int64

### 23.Standardize city names

In [28]:
city_mapping = {
    "Bangalore": "Bengaluru",
    "Banglore": "Bengaluru",
    "bangalore": "Bengaluru"
}

df_clean["city"] = df_clean["city"].replace(city_mapping)

df_clean["city"].value_counts()

city
Los Angeles    12
Seattle        12
New York       11
Chicago        11
Austin         11
Boston         11
Denver         11
Miami          11
Portland       10
London          7
Manchester      7
Bristol         6
Bengaluru       5
Hyderabad       5
Mumbai          4
Delhi           4
Chennai         4
Pune            4
Jaipur          4
Name: count, dtype: int64

### 24.Compare the original and cleaned data

In [29]:
comparison_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "company",
    "city",
    "country",
    "status"
]

comparison = pd.DataFrame({
    "original": df_raw[comparison_columns].stack().reset_index(drop=True),
    "cleaned": df_clean[comparison_columns].stack().reset_index(drop=True)
})

In [30]:
for column in comparison_columns:
    print(f"\n--- {column} ---")
    print("Before:", df_raw[column].unique()[:15])
    print("After: ", df_clean[column].unique()[:15])


--- customer_id ---
Before: <StringArray>
['C0001', 'C0002', 'C0003', 'C0004', 'C0005', 'C0006', 'C0007', 'C0008',
 'C0009', 'C0010', 'C0011', 'C0012', 'C0013', 'C0014', 'C0015']
Length: 15, dtype: str
After:  <StringArray>
['C0001', 'C0002', 'C0003', 'C0004', 'C0005', 'C0006', 'C0007', 'C0008',
 'C0009', 'C0010', 'C0011', 'C0012', 'C0013', 'C0014', 'C0015']
Length: 15, dtype: str

--- first_name ---
Before: <StringArray>
[       'John',     ' Sarah ',     'Michael',       'Emily',       'David',
     'Jessica',      'Daniel',      'Amanda',      'Robert',        'Lisa',
       'James',    'Patricia', 'Christopher',    'Jennifer',     'Matthew']
Length: 15, dtype: str
After:  <StringArray>
[       'John',       'Sarah',     'Michael',       'Emily',       'David',
     'Jessica',      'Daniel',      'Amanda',      'Robert',        'Lisa',
       'James',    'Patricia', 'Christopher',    'Jennifer',     'Matthew']
Length: 15, dtype: str

--- last_name ---
Before: <StringArray>
[   'Smi

### 25.Standardize phone numbers cautiously

In [31]:
df_clean["phone_original"] = df_raw["phone"]

df_clean["phone"] = (
    df_clean["phone"]
    .astype("string")
    .str.strip()
)

df_clean["phone_digits"] = (
    df_clean["phone"]
    .str.replace(r"\D", "", regex=True)
)

df_clean[["customer_id", "country", "phone_original", "phone", "phone_digits"]].head(15)

,customer_id,country,phone_original,phone,phone_digits
0,C0001,United States,-1070,-1070,1070
1,C0002,United States,4155550102,4155550102,4155550102
2,C0003,United States,+1 415 555 0103,+1 415 555 0103,14155550103
3,C0004,United States,NaN,<NA>,<NA>
4,C0005,United States,555-0105,555-0105,5550105
5,C0006,United States,415-555-0106,415-555-0106,4155550106
6,C0007,United States,-1076,-1076,1076
7,C0008,United States,415.555.0108,415.555.0108,4155550108
8,C0009,United States,4155550109,4155550109,4155550109
9,C0010,United States,+1 415 555 0110,+1 415 555 0110,14155550110


### 26 . Flag suspicious phone numbers

In [32]:
df_clean["phone_digits_length"] = df_clean["phone_digits"].str.len()

df_clean["phone_quality"] = np.where(
    df_clean["phone_digits_length"].isna(),
    "Missing",
    np.where(
        df_clean["phone_digits_length"].isin([10, 11, 12]),
        "Check Format",
        "Suspicious"
    )
)

df_clean[["customer_id", "country", "phone", "phone_digits", "phone_quality"]].head(20)

,customer_id,country,phone,phone_digits,phone_quality
0,C0001,United States,-1070,1070,Suspicious
1,C0002,United States,4155550102,4155550102,Check Format
2,C0003,United States,+1 415 555 0103,14155550103,Check Format
3,C0004,United States,<NA>,<NA>,Missing
4,C0005,United States,555-0105,5550105,Suspicious
5,C0006,United States,415-555-0106,4155550106,Check Format
6,C0007,United States,-1076,1076,Suspicious
7,C0008,United States,415.555.0108,4155550108,Check Format
8,C0009,United States,4155550109,4155550109,Check Format
9,C0010,United States,+1 415 555 0110,14155550110,Check Format


### 27.Standardize the signup date

In [33]:
df_clean["signup_date_original"] = df_raw["signup_date"]

df_clean["signup_date"] = (
    df_raw["signup_date"]
    .astype("string")
    .str.strip()
)

df_clean[["customer_id", "signup_date_original"]].head(30)

,customer_id,signup_date_original
0,C0001,15-01-2024
1,C0002,01/22/2024
2,C0003,03-02-2024
3,C0004,12-02-2024
4,C0005,01-03-2024
5,C0006,15-03-2024
6,C0007,20-03-2024
7,C0008,01-04-2024
8,C0009,08-04-2024
9,C0010,15-04-2024


In [34]:
df_clean["signup_date_parsed"] = pd.to_datetime(
    df_clean["signup_date"],
    format="mixed",
    errors="coerce"
)

df_clean[[
    "customer_id",
    "signup_date_original",
    "signup_date_parsed"
]].head(30)

,customer_id,signup_date_original,signup_date_parsed
0,C0001,15-01-2024,2024-01-15
1,C0002,01/22/2024,2024-01-22
2,C0003,03-02-2024,2024-03-02
3,C0004,12-02-2024,2024-12-02
4,C0005,01-03-2024,2024-01-03
5,C0006,15-03-2024,2024-03-15
6,C0007,20-03-2024,2024-03-20
7,C0008,01-04-2024,2024-01-04
8,C0009,08-04-2024,2024-08-04
9,C0010,15-04-2024,2024-04-15


In [35]:
date_errors = df_clean[
    df_clean["signup_date_parsed"].isna()
]

date_errors[[
    "customer_id",
    "signup_date_original"
]]

,customer_id,signup_date_original


### 28.Check the cleaned dataset

In [36]:
print("CLEANED DATA VALIDATION")
print("-" * 40)

print("Total records:", len(df_clean))
print("Unique customer IDs:", df_clean["customer_id"].nunique())
print("Duplicate customer IDs:", df_clean["customer_id"].duplicated().sum())

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nStatus distribution:")
print(df_clean["status"].value_counts(dropna=False))

print("\nCountry distribution:")
print(df_clean["country"].value_counts(dropna=False))

print("\nExact duplicate rows:")
print(df_clean.duplicated().sum())

CLEANED DATA VALIDATION
----------------------------------------
Total records: 150
Unique customer IDs: 150
Duplicate customer IDs: 0

Missing values:
customer_id             0
first_name              0
last_name               0
email                   1
phone                   3
company                 0
city                    0
country                 0
signup_date             0
status                  0
phone_original          3
phone_digits            3
phone_digits_length     3
phone_quality           0
signup_date_original    0
signup_date_parsed      0
dtype: int64

Status distribution:
status
Active      85
Inactive    37
Pending     27
Unknown      1
Name: count, dtype: int64

Country distribution:
country
United States     100
India              30
United Kingdom     20
Name: count, dtype: int64

Exact duplicate rows:
0


### 29.Identify duplicate normalized emails

In [37]:
duplicate_review = df_clean[
    df_clean["email"].notna() &
    df_clean["email"].duplicated(keep=False)
].copy()

duplicate_review = duplicate_review.sort_values("email")

duplicate_review[[
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "company",
    "city",
    "country"
]]

,customer_id,first_name,last_name,email,phone,company,city,country
103,C0104,Ananya,Reddy,ananya.reddy@example.com,<NA>,Bright Solutions,Hyderabad,India
128,C0129,Ananya,Reddy,ananya.reddy@example.com,9876543029,Bright Solutions,Hyderabad,India
100,C0101,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs,Bengaluru,India
120,C0121,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs,Bengaluru,India
67,C0068,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf,Seattle,United States
87,C0088,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf,Seattle,United States
41,C0042,Jonathan,Phillips,jonathan.phillips@example.com,4155550142,NorthStar,Boston,United States
96,C0097,Jonathan,Phillips,jonathan.phillips@example.com,4155550197,CloudNine,Austin,United States
42,C0043,Raymond,Campbell,raymond.campbell@example.com,4155550143,DataWorks,Denver,United States
99,C0100,Raymond,Campbell,raymond.campbell@example.com,4155550200,PixelPoint,Miami,United States


### 30.Final dataset

In [38]:
final_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "company",
    "city",
    "country",
    "signup_date",
    "status"
]

df_final = df_clean[final_columns].copy()

print("Final dataset shape:", df_final.shape)
df_final.head()

Final dataset shape: (150, 10)


,customer_id,first_name,last_name,email,phone,company,city,country,signup_date,status
0,C0001,John,Smith,john.smith@example.com,-1070,Acme Inc,New York,United States,15-01-2024,Active
1,C0002,Sarah,Johnson,sarah.j@example.com,4155550102,BlueSky Ltd,Los Angeles,United States,01/22/2024,Active
2,C0003,Michael,Brown,michael.brown@example.com,+1 415 555 0103,TechNova,Chicago,United States,03-02-2024,Active
3,C0004,Emily,Davis,emily.davis@example.com,<NA>,GreenLeaf,Seattle,United States,12-02-2024,Inactive
4,C0005,David,Wilson,david.wilson@example.com,555-0105,CloudNine,Austin,United States,01-03-2024,Active


### 31.Save the cleaned data

In [39]:
from pathlib import Path

output_folder = Path("../output")
output_folder.mkdir(parents=True, exist_ok=True)

df_final.to_csv(
    output_folder / "customer_contacts_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### 32.Save duplicate records for review

In [40]:
duplicate_review.to_csv(
    output_folder / "duplicate_review.csv",
    index=False
)

print("Duplicate review file saved.")

Duplicate review file saved.


### 33.Generate report

In [41]:
report = f"""
CUSTOMER CONTACT DATA CLEANING REPORT
====================================

Dataset Summary
--------------
Original Records: {len(df_raw)}
Cleaned Records: {len(df_final)}
Original Columns: {len(df_raw.columns)}
Final Columns: {len(df_final.columns)}

Data Quality Checks
-------------------
Duplicate Customer IDs: {df_final['customer_id'].duplicated().sum()}
Exact Duplicate Rows: {df_final.duplicated().sum()}
Missing Emails: {df_final['email'].isna().sum()}
Missing Phones: {df_final['phone'].isna().sum()}
Duplicate Email Records Requiring Review: {len(duplicate_review)}

Standardization Performed
-------------------------
- Removed leading and trailing whitespace
- Standardized first and last name casing
- Converted email addresses to lowercase
- Standardized country names
- Standardized customer status values
- Standardized city name variations

Data Quality Considerations
---------------------------
- Invalid emails require review
- Missing contact information was not fabricated
- Potential duplicate contacts were identified for review
- Phone numbers were inspected for suspicious formats
- Original signup dates were preserved for date validation
- Records were not automatically deleted

Output Files
------------
- customer_contacts_cleaned.csv
- duplicate_review.csv
"""

with open(output_folder / "data_quality_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


CUSTOMER CONTACT DATA CLEANING REPORT

Dataset Summary
--------------
Original Records: 150
Cleaned Records: 150
Original Columns: 10
Final Columns: 10

Data Quality Checks
-------------------
Duplicate Customer IDs: 0
Exact Duplicate Rows: 0
Missing Emails: 1
Missing Phones: 3
Duplicate Email Records Requiring Review: 16

Standardization Performed
-------------------------
- Removed leading and trailing whitespace
- Standardized first and last name casing
- Converted email addresses to lowercase
- Standardized country names
- Standardized customer status values
- Standardized city name variations

Data Quality Considerations
---------------------------
- Invalid emails require review
- Missing contact information was not fabricated
- Potential duplicate contacts were identified for review
- Phone numbers were inspected for suspicious formats
- Original signup dates were preserved for date validation
- Records were not automatically deleted

Output Files
------------
- customer_contac